In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import glob
import os
import numpy as np
from adjustText import adjust_text

# === CONFIG ===
base_dir = "/n/scratch/users/a/adm808/Contrasts"
save_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Final_Outputs_Figures/AD_prediction_stuff_new/Contrasts"
os.makedirs(save_dir, exist_ok=True)

cell_types = ["Opc", "Oli", "Mic", "Ast", "In", "Ex"]

contrast_patterns = {
    "*4vs20*_{ct}.csv": "Clin AD vs Ctrl-Ctrl",
    "*6vs20*_{ct}.csv": "Path AD vs Ctrl-Ctrl",
    "*6vs18*_{ct}.csv": "Clin AD vs Clin Ctrl\n(Path AD cases)",
    "*18vs4*_{ct}.csv": "Path AD vs Path Ctrl\n(Clin AD cases)"
}

lfc_thresh = 0.25
padj_thresh = 0.05

# Style constants
FIGSIZE = (16, 6)
TITLE_FONTSIZE = 10
AX_FONTSIZE = 8
LABEL_FONTSIZE = 6
SUptitle_FONTSIZE = 12
POINT_SIZE = 8
ALPHA = 0.8

# Colors
COLOR_UP = "#d62728"
COLOR_DOWN = "#1f77b4"
COLOR_NONSIG = "#bdbdbd"

def _safe_neglog10(pvals, min_pos=1e-300):
    p = np.array(pvals, dtype=float)
    p = np.where((p > 0) & np.isfinite(p), p, min_pos)
    return -np.log10(p)

# Fixed x-range
X_LIMITS = (-5, 5)

for ct in cell_types:
    dfs = []
    titles = []
    y_max = 0.0

    for pat_template, contrast_title in contrast_patterns.items():
        pat = pat_template.format(ct=ct)
        files = glob.glob(os.path.join(base_dir, pat))
        if not files:
            print(f"[{ct}] No files matched pattern: {pat}")
            dfs.append(None)
            titles.append(contrast_title)
            continue

        df = pd.read_csv(files[0])
        df.columns = [c.lower() for c in df.columns]
        if not {"gene", "log2fc", "padj"}.issubset(df.columns):
            print(f"[{ct}] Missing required columns in {files[0]} — skipping.")
            dfs.append(None)
            titles.append(contrast_title)
            continue

        df = df.dropna(subset=["gene", "log2fc", "padj"]).copy()
        df["neglog10p"] = _safe_neglog10(df["padj"].values)

        # Crop to X_LIMITS
        df = df[(df["log2fc"] >= X_LIMITS[0]) & (df["log2fc"] <= X_LIMITS[1])]

        if not df.empty:
            y_max = max(y_max, float(np.nanmax(df["neglog10p"].values)))

        dfs.append(df)
        titles.append(contrast_title)

    if y_max == 0:
        y_max = 1.0
    ylim = (0, y_max * 1.05 + 0.5)

    fig, axes = plt.subplots(1, 4, figsize=FIGSIZE)
    axes = np.atleast_1d(axes)

    for i in range(4):
        ax = axes[i]

        if dfs[i] is None or dfs[i].empty:
            ax.axis('off')
            ax.set_title(titles[i], fontsize=TITLE_FONTSIZE, pad=8)
            continue

        df = dfs[i]

        sig_mask = (df["padj"] < padj_thresh) & (np.abs(df["log2fc"]) > lfc_thresh)
        up_mask = sig_mask & (df["log2fc"] > lfc_thresh)
        down_mask = sig_mask & (df["log2fc"] < -lfc_thresh)
        ns_mask = ~sig_mask

        ax.scatter(df.loc[ns_mask, "log2fc"], df.loc[ns_mask, "neglog10p"],
                   c=COLOR_NONSIG, s=POINT_SIZE, alpha=ALPHA, edgecolors='none')
        ax.scatter(df.loc[down_mask, "log2fc"], df.loc[down_mask, "neglog10p"],
                   c=COLOR_DOWN, s=POINT_SIZE, alpha=ALPHA, edgecolors='none')
        ax.scatter(df.loc[up_mask, "log2fc"], df.loc[up_mask, "neglog10p"],
                   c=COLOR_UP, s=POINT_SIZE, alpha=ALPHA, edgecolors='none')

        ax.axvline(+lfc_thresh, color="#999999", lw=1, ls="--")
        ax.axvline(-lfc_thresh, color="#999999", lw=1, ls="--")
        ax.axhline(-np.log10(padj_thresh), color="#999999", lw=1, ls="--")

        # --- Top labels ---
        sig_df = df.loc[sig_mask, ["gene", "log2fc", "neglog10p", "padj"]].copy()

        # Top 10 by |log2FC|
        top_lfc = sig_df.reindex(sig_df["log2fc"].abs().sort_values(ascending=False).index)[:10]

        # Top 5 by smallest p-value
        top_p = sig_df.sort_values("padj", ascending=True)[:5]

        # Combine sets, keeping unique genes
        label_df = pd.concat([top_lfc, top_p]).drop_duplicates(subset="gene")

        texts = []
        for _, row in label_df.iterrows():
            texts.append(ax.text(row["log2fc"], row["neglog10p"], str(row["gene"]),
                                 fontsize=LABEL_FONTSIZE, ha="left", va="center"))

        adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))

        ax.set_xlim(*X_LIMITS)
        ax.set_ylim(*ylim)
        ax.set_title(titles[i], fontsize=TITLE_FONTSIZE, pad=8)
        ax.set_xlabel("log2FC", fontsize=AX_FONTSIZE)
        if i == 0:
            ax.set_ylabel("-log10(adj. p)", fontsize=AX_FONTSIZE)
        ax.tick_params(labelsize=AX_FONTSIZE)

    fig.suptitle(f"{ct}: Volcano plots for 4 contrasts", fontsize=SUptitle_FONTSIZE)
    fig.subplots_adjust(top=0.88, right=0.98, left=0.06, bottom=0.12, wspace=0.25)

    save_path = os.path.join(save_dir, f"{ct}_4_contrasts_VOLCANO_topLFC_topP.png")
    fig.savefig(save_path, dpi=300)
    plt.show()
    print(f"Saved {save_path}")

In [ ]:
import pandas as pd
import gseapy as gp
import glob
import os
import numpy as np

# Path to your Microglia cognitive resilience file
base_dir = "/n/scratch/users/a/adm808/Contrasts"
save_path = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Final_Outputs_Figures/AD_prediction_stuff_new/Contrasts/Mic"
files = glob.glob(os.path.join(base_dir, "*6vs18*_Mic.csv"))
if not files:
    raise FileNotFoundError("No *6vs18*_Mic.csv file found.")
fpath = files[0]

# Load file
df = pd.read_csv(fpath)
df.columns = [c.lower() for c in df.columns]

if not {'gene', 'log2fc'}.issubset(df.columns):
    raise ValueError("CSV must have 'gene' and 'log2fc' columns.")

# Rank all genes by log2FC
ranked_genes = df[['gene', 'log2fc']].dropna()
ranked_genes = ranked_genes.sort_values('log2fc', ascending=False)
rnk_path = os.path.join(base_dir, "mic_6vs18_ranked.rnk")
ranked_genes.to_csv(rnk_path, sep="\t", index=False, header=False)

# Run GSEA (GO:BP)
pre_res = gp.prerank(
    rnk=rnk_path,
    gene_sets='GO_Biological_Process_2021',
    processes=4,
    min_size=10,
    max_size=500,
    permutation_num=1000,
    outdir=None,
    seed=42,
    verbose=False
)

# Show top pathways
res_df = pre_res.res2d
# print("\n=== Top GSEA pathways ===")
# print(res_df.head(10))

import matplotlib.pyplot as plt

# Sort by NES magnitude and take top 10
top_res = res_df.sort_values("NES", key=abs, ascending=False).head(10)
top_res["Term"] = top_res["Term"].str.replace(r"\s*\(GO:\d+\)", "", regex=True)
top_res["Term"] = top_res["Term"].str.capitalize()


plt.figure(figsize=(10, 4))
bars = plt.barh(top_res["Term"], top_res["NES"],
                color=top_res["NES"].apply(lambda x: 'red' if x > 0 else 'blue'))
plt.xlabel("NES (Normalized Enrichment Score)")
plt.title("Top Microglia Pathways between Pathology Postiive-Clinical AD and Pathology Positive-Clinical Control")
plt.gca().invert_yaxis()  # highest on top
plt.axvline(0, color='black', linewidth=0.8)


# dynamic right limit so bars aren’t crammed
right = np.ceil(top_res["NES"].max() / 0.25) * 0.25
plt.xlim(0, right)
plt.xticks(np.arange(0, right + 1e-9, 0.25))

save_path = os.path.join(save_path, "mic_6vs18_top_pathways.png")

plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd
import gseapy as gp
import glob
import os
import numpy as np

# Path to your Microglia cognitive resilience file
base_dir = "/n/scratch/users/a/adm808/Contrasts"
save_path = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Final_Outputs_Figures/AD_prediction_stuff_new/Contrasts/Oli"
files = glob.glob(os.path.join(base_dir, "*6vs18*_Oli.csv"))
if not files:
    raise FileNotFoundError("No *6vs18*_Oli.csv file found.")
fpath = files[0]

# Load file
df = pd.read_csv(fpath)
df.columns = [c.lower() for c in df.columns]

if not {'gene', 'log2fc'}.issubset(df.columns):
    raise ValueError("CSV must have 'gene' and 'log2fc' columns.")

# Rank all genes by log2FC
ranked_genes = df[['gene', 'log2fc']].dropna()
ranked_genes = ranked_genes.sort_values('log2fc', ascending=False)
rnk_path = os.path.join(base_dir, "Oli_6vs18_ranked.rnk")
ranked_genes.to_csv(rnk_path, sep="\t", index=False, header=False)

# Run GSEA (GO:BP)
pre_res = gp.prerank(
    rnk=rnk_path,
    gene_sets='GO_Biological_Process_2021',
    processes=4,
    min_size=10,
    max_size=500,
    permutation_num=1000,
    outdir=None,
    seed=42,
    verbose=False
)

# Show top pathways
res_df = pre_res.res2d
print("\n=== Top GSEA pathways ===")
print(res_df.head(10))


import matplotlib.pyplot as plt

# Sort by NES magnitude and take top 10
top_res = res_df.sort_values("NES", key=abs, ascending=False).head(10)
top_res["Term"] = top_res["Term"].str.replace(r"\s*\(GO:\d+\)", "", regex=True)
top_res["Term"] = top_res["Term"].str.capitalize()

plt.figure(figsize=(10, 4))
bars = plt.barh(top_res["Term"], top_res["NES"],
                color=top_res["NES"].apply(lambda x: 'red' if x > 0 else 'blue'))
plt.xlabel("NES (Normalized Enrichment Score)")
plt.title("Top Oligodendrocyte Pathways between Pathology Postiive-Clinical AD and Pathology Positive-Clinical Control")
plt.gca().invert_yaxis()  # highest on top
plt.axvline(0, color='black', linewidth=0.8)

# dynamic right limit so bars aren’t crammed
right = np.ceil(top_res["NES"].max() / 0.25) * 0.25
plt.xlim(0, right)
plt.xticks(np.arange(0, right + 1e-9, 0.25))

save_path = os.path.join(save_path, "oli_6vs18_top_pathways.png")

plt.savefig(save_path, dpi=300, bbox_inches='tight')

plt.show()